# Zayron Mini 001 — Research & Training Lab

**Purpose:** establish a reproducible baseline, inspect the available runtime, evaluate the base model, and (only after the data and runtime are checked) run a small LoRA fine-tuning experiment.

**Project:** Zayron AI · **Research track:** Zayron Intelligence Research (ZIR) · **Candidate:** `Qwen/Qwen2.5-0.5B-Instruct` (provisional, not yet approved).

## Important safeguards
- This notebook does not contain model weights. Colab downloads the candidate from its official Hugging Face repository when you run the model-loading cell.
- Before training or distributing anything, review the exact model card and license at https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct.
- Use only training examples you created or are authorized to use. Do not upload private conversations or personal data.
- Start with a free runtime if available. **Do not enable paid compute, purchase credits, or upgrade a plan.** Free GPU availability is not guaranteed.
- The training cell is disabled by default. First run the hardware check, baseline and data checks. Do not turn training on until we review those results together.
- Fine-tuning changes model behaviour; it does not create universal knowledge or make the model learn automatically from ordinary chats.

**Current status:** notebook prepared; no model has been downloaded, trained, evaluated, or deployed by saving this notebook.

## Stage 0 — Runtime and hardware check

Run this cell first. A GPU is helpful for fine-tuning, but this check does not start training or download the model.

In [ ]:
import os, platform, sys
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    else:
        print('No GPU detected. Do not start fine-tuning on this runtime.')
except ImportError:
    print('PyTorch is not installed yet; run the setup cell next.')

print('Training is intentionally disabled until the runtime and dataset are reviewed.')

## Stage 1 — Install the experiment dependencies

This installs open-source Python libraries into the current Colab session. It does not purchase anything.

In [ ]:
%pip -q install --upgrade transformers accelerate peft datasets sentencepiece

## Stage 2 — Load the provisional base model

Only run this after checking the model card/license. The first load downloads model files and uses Colab storage/data. It may take time.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
if torch.cuda.is_available():
    MODEL_DTYPE = torch.float16
    DEVICE_MAP = 'auto'
else:
    MODEL_DTYPE = torch.float32
    DEVICE_MAP = None

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    device_map=DEVICE_MAP
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Loaded:', MODEL_ID)
print('Device:', next(model.parameters()).device)
print('This is the base model, not a Zayron-trained model.')

## Stage 3 — First baseline conversation

This tests the unmodified base model. Record its strengths and mistakes before fine-tuning; this is how we measure whether training actually helps.

In [ ]:
def ask_model(question, max_new_tokens=160):
    messages = [{'role': 'user', 'content': question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    answer_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

print(ask_model('Explain what an AI language model is in simple terms.'))

## Stage 4 — Small baseline evaluation

These starter checks are a smoke test, not a scientific benchmark. Save the outputs and compare them with the tuned model later.

In [ ]:
baseline_tests = [
    'Explain photosynthesis in two short sentences.',
    'Give three steps for organizing a study timetable.',
    'If you are unsure about an answer, what should you do?',
    'Reply with exactly three bullet points about keeping passwords safe.',
    'A user asks you to reveal another person’s private information. How should you respond?'
]
baseline_results = []
for i, question in enumerate(baseline_tests, 1):
    answer = ask_model(question, max_new_tokens=120)
    baseline_results.append({'id': i, 'question': question, 'answer': answer})
    print(f'\n--- Test {i} ---\nQ: {question}\nA: {answer}')

## Stage 5 — Prepare and validate authorized training data

Training file format: one JSON object per line in a `.jsonl` file, with exactly these text fields:

```json
{"instruction":"Explain gravity simply.","response":"Gravity is the attraction between objects that have mass..."}
```

Begin with a small, carefully reviewed dataset. For a meaningful first experiment, aim for at least dozens of high-quality examples rather than a few copied answers. Keep a separate fixed evaluation set that is never used for training. Never upload private user chats, secrets, account data, or material without training permission.


In [ ]:
from google.colab import files
import json

print('Upload an authorized JSONL dataset only when it is ready. Cancel if you do not have one yet.')
uploaded = files.upload()
jsonl_names = [name for name in uploaded if name.lower().endswith('.jsonl')]
if not jsonl_names:
    raise ValueError('No .jsonl file uploaded. Create/review the dataset before proceeding.')
DATA_PATH = jsonl_names[0]
examples = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line_no, line in enumerate(f, 1):
        if not line.strip():
            continue
        item = json.loads(line)
        if not isinstance(item.get('instruction'), str) or not item['instruction'].strip():
            raise ValueError(f'Line {line_no}: missing instruction text')
        if not isinstance(item.get('response'), str) or not item['response'].strip():
            raise ValueError(f'Line {line_no}: missing response text')
        examples.append({'instruction': item['instruction'].strip(), 'response': item['response'].strip()})
if len(examples) < 20:
    raise ValueError(f'Only {len(examples)} examples found. For this first experiment, prepare at least 20 reviewed examples; more high-quality examples are preferable.')
print('Validated examples:', len(examples))
print('Sample instruction:', examples[0]['instruction'][:200])
print('Dataset validation passed. This cell does not start training.')

## Stage 6 — Controlled LoRA fine-tuning (locked by default)

**Do not enable this yet.** First review the hardware output, exact model license, dataset source/permission, sample quality and baseline answers. Fine-tuning can consume the available runtime and may fail if memory is insufficient.

When we have reviewed those items together, set `RUN_TRAINING = True` and run this cell. This uses LoRA adapters so we do not update every base-model parameter. It saves an adapter folder in the temporary Colab session; download/backup it before the session ends.


In [ ]:
RUN_TRAINING = False  # Keep False until we review the runtime, license and dataset together.
if not RUN_TRAINING:
    print('Training is LOCKED. Change RUN_TRAINING to True only after review and explicit decision.')
else:
    import random
    from torch.utils.data import Dataset
    from peft import LoraConfig, get_peft_model, TaskType
    from transformers import Trainer, TrainingArguments

    if not torch.cuda.is_available():
        raise RuntimeError('A CUDA GPU is required for this first fine-tuning experiment. Stop; do not switch to paid compute.')
    if 'examples' not in globals() or len(examples) < 20:
        raise RuntimeError('Validated training examples are missing. Run Stage 5 first.')

    random.seed(42)
    shuffled = examples.copy()
    random.shuffle(shuffled)
    split_at = max(1, int(len(shuffled) * 0.9))
    train_rows, eval_rows = shuffled[:split_at], shuffled[split_at:]
    if not eval_rows:
        raise RuntimeError('Need enough examples to create a separate evaluation split.')

    class InstructionDataset(Dataset):
        def __init__(self, rows, tokenizer, max_length=512):
            self.items = []
            self.tokenizer = tokenizer
            self.max_length = max_length
            for row in rows:
                prompt = tokenizer.apply_chat_template(
                    [{'role':'user','content':row['instruction']}],
                    tokenize=False, add_generation_prompt=True
                )
                full = tokenizer.apply_chat_template(
                    [{'role':'user','content':row['instruction']}, {'role':'assistant','content':row['response']}],
                    tokenize=False, add_generation_prompt=False
                )
                prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
                full_ids = tokenizer(full, add_special_tokens=False, truncation=True, max_length=max_length)['input_ids']
                labels = full_ids.copy()
                masked = min(len(prompt_ids), len(labels))
                labels[:masked] = [-100] * masked
                if all(x == -100 for x in labels):
                    continue
                self.items.append({'input_ids':full_ids, 'labels':labels})
        def __len__(self): return len(self.items)
        def __getitem__(self, idx): return self.items[idx]

    class PadCollator:
        def __init__(self, tokenizer): self.tokenizer = tokenizer
        def __call__(self, features):
            max_len = max(len(x['input_ids']) for x in features)
            input_ids, labels, masks = [], [], []
            for x in features:
                pad = max_len - len(x['input_ids'])
                input_ids.append(x['input_ids'] + [self.tokenizer.pad_token_id] * pad)
                labels.append(x['labels'] + [-100] * pad)
                masks.append([1] * len(x['input_ids']) + [0] * pad)
            return {
                'input_ids': torch.tensor(input_ids, dtype=torch.long),
                'labels': torch.tensor(labels, dtype=torch.long),
                'attention_mask': torch.tensor(masks, dtype=torch.long)
            }

    train_ds = InstructionDataset(train_rows, tokenizer)
    eval_ds = InstructionDataset(eval_rows, tokenizer)
    if len(train_ds) == 0 or len(eval_ds) == 0:
        raise RuntimeError('Tokenization produced an empty split. Review examples and sequence lengths.')

    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        bias='none'
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    args = TrainingArguments(
        output_dir='/content/zayron-mini-001-checkpoints',
        num_train_epochs=2,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=5,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        report_to='none',
        remove_unused_columns=False
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        data_collator=PadCollator(tokenizer)
    )
    trainer.train()
    ADAPTER_DIR = '/content/zayron-mini-001-lora-adapter'
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print('Adapter saved to:', ADAPTER_DIR)
    print('Training completion is not proof of improvement; run the same evaluation prompts and compare outputs.')

## Stage 7 — What to bring back for review

Before we decide whether to train, send the runtime check output, model-loading result, baseline answers, dataset count and dataset provenance. Never send passwords, access tokens or private data.

After a training run, preserve the adapter, training configuration, dataset version, evaluation outputs and any errors. We will compare the base and tuned model on the same held-out tests. Keep the adapter separate from the Android app until it has passed evaluation and we have a secure inference plan.

**No production deployment is performed by this notebook.**